In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

# 5 Bengaluru Locations
locations = [
    {"name": "MG Road",         "lat": 12.9757, "lon": 77.6011},
    {"name": "Whitefield",      "lat": 12.9698, "lon": 77.7500},
    {"name": "Jayanagar",       "lat": 12.9308, "lon": 77.5838},
    {"name": "Hebbal",          "lat": 13.0350, "lon": 77.5970},
    {"name": "Electronic City", "lat": 12.8399, "lon": 77.6770},
]

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AQ_URL      = "https://air-quality-api.open-meteo.com/v1/air-quality"

In [2]:
weather_all = []

for loc in locations:
    print(f"Fetching weather: {loc['name']} ...", end=" ")

    params = {
        "latitude":   loc["lat"],
        "longitude":  loc["lon"],
        "hourly":     "temperature_2m,relative_humidity_2m",
        "timezone":   "Asia/Kolkata",
        "past_days":  21,
    }

    try:
        r = requests.get(WEATHER_URL, params=params, timeout=20)
        df = pd.DataFrame(r.json()["hourly"])
        df.rename(columns={
            "temperature_2m":       "temperature",
            "relative_humidity_2m": "humidity"
        }, inplace=True)
        df["time"]      = pd.to_datetime(df["time"])
        df["location"]  = loc["name"]
        df["latitude"]  = loc["lat"]
        df["longitude"] = loc["lon"]
        weather_all.append(df)
        print(f"✓  {len(df)} rows")
    except Exception as e:
        print(f"✗  Failed — {e}")

    time.sleep(1)

# Combine & Save
weather_df = pd.concat(weather_all, ignore_index=True)
weather_df = weather_df[["location","latitude","longitude","time","temperature","humidity"]]
weather_df.to_csv("bengaluru_weather.csv", index=False)

print("\nSaved: bengaluru_weather.csv")
print("Shape:", weather_df.shape)
weather_df.head()

Fetching weather: MG Road ... ✓  672 rows
Fetching weather: Whitefield ... ✓  672 rows
Fetching weather: Jayanagar ... ✓  672 rows
Fetching weather: Hebbal ... ✓  672 rows
Fetching weather: Electronic City ... ✓  672 rows

Saved: bengaluru_weather.csv
Shape: (3360, 6)


,location,latitude,longitude,time,temperature,humidity
0,MG Road,12.9757,77.6011,2026-07-27 00:00:00,21.8,86
1,MG Road,12.9757,77.6011,2026-07-27 01:00:00,21.4,88
2,MG Road,12.9757,77.6011,2026-07-27 02:00:00,21.2,88
3,MG Road,12.9757,77.6011,2026-07-27 03:00:00,20.8,90
4,MG Road,12.9757,77.6011,2026-07-27 04:00:00,20.7,91


In [3]:
aq_all = []

for loc in locations:
    print(f"Fetching air quality: {loc['name']} ...", end=" ")

    params = {
        "latitude":   loc["lat"],
        "longitude":  loc["lon"],
        "hourly":     "pm10,pm2_5,carbon_monoxide",
        "timezone":   "Asia/Kolkata",
        "past_days":  30,
    }

    try:
        r = requests.get(AQ_URL, params=params, timeout=20)
        df = pd.DataFrame(r.json()["hourly"])
        df["time"]      = pd.to_datetime(df["time"])
        df["location"]  = loc["name"]
        df["latitude"]  = loc["lat"]
        df["longitude"] = loc["lon"]
        aq_all.append(df)
        print(f"✓  {len(df)} rows")
    except Exception as e:
        print(f"✗  Failed — {e}")

    time.sleep(1)

# Combine & Save
aq_df = pd.concat(aq_all, ignore_index=True)
aq_df = aq_df[["location","latitude","longitude","time","pm10","pm2_5","carbon_monoxide"]]
aq_df.to_csv("bengaluru_air_quality.csv", index=False)

print("\nSaved: bengaluru_air_quality.csv")
print("Shape:", aq_df.shape)
aq_df.head()

Fetching air quality: MG Road ... ✓  840 rows
Fetching air quality: Whitefield ... ✓  840 rows
Fetching air quality: Jayanagar ... ✓  840 rows
Fetching air quality: Hebbal ... ✓  840 rows
Fetching air quality: Electronic City ... ✓  840 rows

Saved: bengaluru_air_quality.csv
Shape: (4200, 7)


,location,latitude,longitude,time,pm10,pm2_5,carbon_monoxide
0,MG Road,12.9757,77.6011,2026-07-18 00:00:00,7.7,6.1,251.0
1,MG Road,12.9757,77.6011,2026-07-18 01:00:00,6.5,5.1,201.0
2,MG Road,12.9757,77.6011,2026-07-18 02:00:00,6.4,4.6,165.0
3,MG Road,12.9757,77.6011,2026-07-18 03:00:00,7.8,5.1,148.0
4,MG Road,12.9757,77.6011,2026-07-18 04:00:00,8.6,5.5,145.0


In [4]:
BASE_URL = "https://books.toscrape.com/catalogue/"
url      = "https://books.toscrape.com/catalogue/page-1.html"

# Rating words to numbers
rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}

books = []
page  = 1

while url:
    print(f"Scraping page {page} ...", end=" ")
    r    = requests.get(url, timeout=20)
    soup = BeautifulSoup(r.text, "html.parser")

    for book in soup.select("article.product_pod"):
        title  = book.h3.a["title"]
        price  = book.select_one("p.price_color").text.strip()
        rating = rating_map.get(book.p["class"][1], 0)
        books.append({"title": title, "price": price, "rating": rating})

    # Next page
    next_btn = soup.select_one("li.next a")
    url      = BASE_URL + next_btn["href"] if next_btn else None
    page    += 1
    time.sleep(0.5)

print(f"\nTotal books scraped: {len(books)}")

# Save
books_df = pd.DataFrame(books)
books_df.to_csv("books_data.csv", index=False)

print("Saved: books_data.csv")
print("Shape:", books_df.shape)
books_df.head()

Scraping page 1 ... Scraping page 2 ... Scraping page 3 ... Scraping page 4 ... Scraping page 5 ... Scraping page 6 ... Scraping page 7 ... Scraping page 8 ... Scraping page 9 ... Scraping page 10 ... Scraping page 11 ... Scraping page 12 ... Scraping page 13 ... Scraping page 14 ... Scraping page 15 ... Scraping page 16 ... Scraping page 17 ... Scraping page 18 ... Scraping page 19 ... Scraping page 20 ... Scraping page 21 ... Scraping page 22 ... Scraping page 23 ... Scraping page 24 ... Scraping page 25 ... Scraping page 26 ... Scraping page 27 ... Scraping page 28 ... Scraping page 29 ... Scraping page 30 ... Scraping page 31 ... Scraping page 32 ... Scraping page 33 ... Scraping page 34 ... Scraping page 35 ... Scraping page 36 ... Scraping page 37 ... Scraping page 38 ... Scraping page 39 ... Scraping page 40 ... Scraping page 41 ... Scraping page 42 ... Scraping page 43 ... Scraping page 44 ... Scraping page 45 ... Scraping page 46 ... Scraping page 47 ... Scraping page 48 ... S

,title,price,rating
0,A Light in the Attic,Â£51.77,3
1,Tipping the Velvet,Â£53.74,1
2,Soumission,Â£50.10,1
3,Sharp Objects,Â£47.82,4
4,Sapiens: A Brief History of Humankind,Â£54.23,5


In [5]:
from pathlib import Path

print("Files collected in this lab:\n")
for f in sorted(Path(".").glob("*.csv")):
    print(f"  {f.name:30s}  {f.stat().st_size:>10,} bytes")

Files collected in this lab:

  bengaluru_air_quality.csv          256,792 bytes
  bengaluru_weather.csv              182,840 bytes
  books_data.csv                      53,455 bytes


In [7]:
import os
print(os.getcwd())

C:\Users\vivek
